# Árbol de decisión para predecir aprobación en Carreras Universitarias con Pipeline + MLflow

Este notebook entrena un modelo de árbol de decisión usando el dataset `estudiantes.csv`.

```text
datos originales → preprocesamiento → árbol de decisión
```

De esta manera, Flask o Streamlit pueden enviar las columnas originales del dataset, sin tener que enviar columnas codificadas manualmente.


## 1. Importación de librerías

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from mlflow.models.signature import infer_signature

import mlflow
import mlflow.sklearn


## 2. Configuración de MLflow



In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:9090")
mlflow.set_experiment("carreras_arbol_pipeline")


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1778897915890, experiment_id='3', last_update_time=1778897915890, lifecycle_stage='active', name='carreras_arbol_pipeline', tags={}, trace_location=None, workspace='default'>

## 3. Carga del dataset

In [3]:
df = pd.read_csv("data/estudiantes.csv", delimiter=",")
df.head()


,carrera,modalidad,beca,edad,promedio,asistencias,aprobado
0,Industrial,Presencial,Si,29,5.8,64,Si
1,Industrial,Hibrida,Si,27,6.6,51,No
2,Arquitectura,Presencial,Si,29,8.2,84,Si
3,Economia,Presencial,Si,29,6.6,67,No
4,Economia,Presencial,Si,24,5.1,72,No


In [4]:
print("Filas y columnas:", df.shape)
df.info()


Filas y columnas: (5000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   carrera      5000 non-null   object 
 1   modalidad    5000 non-null   object 
 2   beca         5000 non-null   object 
 3   edad         5000 non-null   int64  
 4   promedio     5000 non-null   float64
 5   asistencias  5000 non-null   int64  
 6   aprobado     5000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 273.6+ KB


## 4. Preparación de datos

La variable objetivo `y` se transforma:

- `yes` → `1`
- `no` → `0`

Además, se elimina `promedio` de las variables predictoras porque normalmente se conoce después de realizada la llamada. Por tanto, puede generar fuga de información si se usa para predecir.


In [5]:
df["aprobado"] = df["aprobado"].map({"Si": 1, "No": 0})

X = df.drop(columns=["aprobado", "promedio"])
y = df["aprobado"]

print("Columnas originales usadas para entrenamiento:")
print(X.columns.tolist())

print("\nDistribución de la variable objetivo:")
print(y.value_counts())


Columnas originales usadas para entrenamiento:
['carrera', 'modalidad', 'beca', 'edad', 'asistencias']

Distribución de la variable objetivo:
aprobado
0    2950
1    2050
Name: count, dtype: int64


## 5. Identificación de columnas categóricas y numéricas

El pipeline hará automáticamente el `OneHotEncoder` para las columnas categóricas y dejará pasar las columnas numéricas.


In [6]:
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
columnas_numericas = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Columnas categóricas:")
print(columnas_categoricas)

print("\nColumnas numéricas:")
print(columnas_numericas)


Columnas categóricas:
['carrera', 'modalidad', 'beca']

Columnas numéricas:
['edad', 'asistencias']


## 6. División de datos

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (4000, 5)
Prueba: (1000, 5)


## 7. Creación del Pipeline

Este es el punto clave para producción.

El `Pipeline` contiene:

1. `preprocessor`: transforma variables categóricas con `OneHotEncoder`.
2. `modelo`: entrena el árbol de decisión.

Cuando guardamos este `Pipeline` en MLflow, Streamlit podrá enviar datos originales como `carrera`, `modalidad`, `beca`, etc.


In [8]:
# Parámetros del árbol
criterion = "gini"
max_depth = 5
min_samples_split = 100
min_samples_leaf = 50
random_state = 42

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas)
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("modelo", DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        ))
    ]
)

pipeline


,steps,"[('preprocessor', ...), ('modelo', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 8. Entrenamiento, evaluación y registro en MLflow

El modelo se registrará con el nombre:

```text
clase06
```

Luego se podrá cargar desde Streamlit con:

```python
mlflow.sklearn.load_model("models:/arbolesCarrera01/1")
```


In [9]:
with mlflow.start_run(run_name="arbol_pipeline_base") as run:

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Parámetros del algoritmo
    mlflow.log_param("modelo", "DecisionTreeClassifier")
    mlflow.log_param("criterion", criterion)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("random_state", random_state)

    # Parámetros del experimento
    mlflow.log_param("dataset", "estudiantes.csv")
    mlflow.log_param("target", "aprobado")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("stratify", True)
    mlflow.log_param("removed_column", "promedio")
    mlflow.log_param("categorical_encoder", "OneHotEncoder")
    mlflow.log_param("handle_unknown", "ignore")

    # Métricas
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    
    # Firma del modelo: ayuda a documentar columnas esperadas
    input_example = X_train.head(5)
    signature = infer_signature(input_example, pipeline.predict(input_example))

    # Guardar y registrar el pipeline completo
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="modelo_carrera_pipeline",
        registered_model_name="arbolesCarrera01",
        input_example=input_example,
        signature=signature
    )

    print("RUN ID:", run.info.run_id)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("\nReporte de clasificación:")


C:\Users\Luis Egas\Documents\MAESTRIA IA 2026\herramientasIA\entorno05\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/16 14:29:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/16 14:29:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or clou

Registered model 'arbolesCarrera01' already exists. Creating a new version of this model...
2026/05/16 14:29:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: arbolesCarrera01, version 3
Created version '3' of model 'arbolesCarrera01'.


RUN ID: b605163298eb4e268aff725696fc660b
Accuracy: 0.693
Precision: 0.6102783725910065
Recall: 0.6951219512195121
F1: 0.6499429874572406

Reporte de clasificación:
🏃 View run arbol_pipeline_base at: http://127.0.0.1:9090/#/experiments/3/runs/b605163298eb4e268aff725696fc660b
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/3
